# 📓 Lab 06 — Fine-tuning de DistilBERT con Hugging Face

**Curso:** Deep Learning: de los fundamentos a los Transformers · **Sesión 4 · Laboratorio 4**
**Material del aula:** [Sesión 4](../sesiones/04-hugging-face-proyecto.md) ·
**Config:** [`configs/transformer.yaml`](../configs/transformer.yaml)

Flujo completo: checkpoint → tokenizer → dataset → `Trainer` → evaluación →
análisis de errores → guardado reproducible.

> ⏱️ **Presupuesto:** en GPU, ~10 min; en CPU, activa el modo `quick_demo`
> (subconjunto) en la sección 3. Primero valida que el pipeline COMPLETO
> funciona con datos pequeños; después escala.

In [ ]:
from __future__ import annotations

import json
import sys
sys.path.append('..')
from pathlib import Path

import numpy as np
import torch

from src.utils import seed_everything

seed_everything(42)
print('GPU disponible:', torch.cuda.is_available())

## 1. Exploración con `pipeline` (5 minutos, cero código de entrenamiento)

Validar el contrato de entrada/salida de la tarea antes de invertir en
fine-tuning.

In [ ]:
from transformers import pipeline

# Un checkpoint YA fine-tuned para sentimiento (para explorar la tarea)
classifier = pipeline(
    task='sentiment-analysis',
    model='distilbert/distilbert-base-uncased-finetuned-sst-2-english',
)

ejemplos = [
    'The course explains attention clearly and the labs are useful.',
    'The installation process was confusing and nothing worked.',
]
for resultado in classifier(ejemplos):
    print(resultado)

# ⚠️ pipeline explora; NO sustituye leer la model card:
# https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english

## 2. Tokenizer: el contrato con el modelo

**Tokenizer y modelo deben venir del MISMO checkpoint.** Inspeccionemos
qué produce.

In [ ]:
from transformers import AutoTokenizer

model_name = 'distilbert/distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

salida = tokenizer('Deep learning is fascinating!')
print('input_ids     :', salida['input_ids'])
print('attention_mask:', salida['attention_mask'])
print('tokens        :', tokenizer.convert_ids_to_tokens(salida['input_ids']))
# Observa: [CLS] al inicio, [SEP] al final, y 'fascinating' → subwords

## 3. Dataset: rotten_tomatoes

Reseñas de cine, clasificación binaria, splits listos. Pequeño y perfecto
para un laboratorio.

In [ ]:
from datasets import load_dataset

raw = load_dataset('rotten_tomatoes')
print(raw)
print('\nEjemplo:', raw['train'][0])

# ── MODO DEMO RÁPIDA (CPU / poco tiempo): descomenta estas líneas ──
# raw['train'] = raw['train'].shuffle(seed=42).select(range(2500))
# raw['validation'] = raw['validation'].select(range(500))
# raw['test'] = raw['test'].select(range(500))

In [ ]:
# Tokenizar todo el dataset (map procesa por batches, es rápido)
def tokenize_batch(batch):
    # truncation corta a max_length; el PADDING se hace después,
    # por batch (dynamic padding) — no aquí, sería desperdicio
    return tokenizer(batch['text'], truncation=True, max_length=256)

tokenized = raw.map(tokenize_batch, batched=True)

from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(tokenized['train'])

## 4. Modelo y métricas

`AutoModelForSequenceClassification` añade una cabeza de clasificación
(inicializada al azar — el warning que verás es ESPERADO: esa cabeza es
justo lo que vamos a entrenar).

In [ ]:
from transformers import AutoModelForSequenceClassification
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: 'NEGATIVE', 1: 'POSITIVE'},
    label2id={'NEGATIVE': 0, 'POSITIVE': 1},
)

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    """accuracy + macro-F1 (la métrica principal del curso)."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions,
                                       references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels,
                           average='macro')
    return {'accuracy': accuracy['accuracy'], 'macro_f1': f1['f1']}

## 5. Fine-tuning con `Trainer`

Cada argumento está justificado en la
[Sesión 4 §4](../sesiones/04-hugging-face-proyecto.md#4-fine-tuning-con-trainer).
Claves: LR pequeño (2e-5) para no destruir lo preentrenado, warmup, y
selección del mejor checkpoint por **macro-F1 en validation**.

In [ ]:
from transformers import (EarlyStoppingCallback, Trainer,
                          TrainingArguments)

output_dir = Path('../artifacts/distilbert-rotten-tomatoes')
output_dir.mkdir(parents=True, exist_ok=True)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir=str(output_dir),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 6. Evaluación final (test: una sola vez)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

validation_metrics = trainer.evaluate(tokenized['validation'])
print('Validation:', {k: round(v, 4) for k, v in validation_metrics.items()
                      if 'loss' in k or 'accuracy' in k or 'f1' in k})

test_output = trainer.predict(tokenized['test'])
predictions = np.argmax(test_output.predictions, axis=-1)
labels = np.asarray(tokenized['test']['label'])

print(classification_report(labels, predictions,
                            target_names=['NEGATIVE', 'POSITIVE'], digits=3))
print(confusion_matrix(labels, predictions))

## 7. Análisis de errores con taxonomía

Los 20 errores de mayor confianza, para etiquetar a mano con la
[taxonomía del curso](../sesiones/04-hugging-face-proyecto.md#taxonomía-de-errores-del-curso):
negación / ironía / ambigüedad / etiqueta dudosa / truncation / fuera de dominio.

In [ ]:
import pandas as pd

probabilities = torch.tensor(test_output.predictions).softmax(dim=-1).numpy()
confidence = probabilities.max(axis=1)

errores = pd.DataFrame({
    'text': raw['test']['text'],
    'label': labels,
    'prediction': predictions,
    'confidence': confidence,
})
errores = errores[errores['label'] != errores['prediction']]
errores = errores.sort_values('confidence', ascending=False)

# Columna para tu etiquetado manual:
errores['error_type'] = ''
errores.head(20)[['confidence', 'label', 'prediction', 'text']]

**Tu análisis:** etiqueta `error_type` para los 20 primeros y responde:
¿cuál es la categoría dominante y qué UNA acción recomiendas
(recolectar datos / revisar etiquetas / ajustar longitud / aceptar el límite)?

*Escribe aquí...*

## 8. Guardar de forma reproducible

Modelo + tokenizer + métricas, juntos. La demo Gradio
([`app/gradio_app.py`](../app/gradio_app.py)) carga desde esta ruta.

In [ ]:
trainer.save_model(str(output_dir / 'best_model'))
tokenizer.save_pretrained(str(output_dir / 'best_model'))

with open(output_dir / 'test_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(test_output.metrics, f, indent=2)

print('Guardado en', output_dir / 'best_model')
# ⚠️ artifacts/ está en .gitignore: los checkpoints NO van al repo.
# Para publicar en el Hub: revisar autenticación, licencia y model card ANTES.

## 9. 🎁 Extensión: PEFT/LoRA

$$W' = W + \Delta W, \qquad \Delta W \approx BA \;\; (r \ll d)$$

Entrena <1% de los parámetros. Inspecciona `model.named_modules()` para
confirmar los nombres de los módulos objetivo (específicos de cada arquitectura).

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

base = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_lin', 'v_lin'],   # específico de DistilBERT
)

peft_model = get_peft_model(base, lora_config)
peft_model.print_trainable_parameters()
# LoRA reduce parámetros entrenables y artifacts.
# NO garantiza calidad ni elimina la validación: eso sigue siendo tu trabajo.

## 📝 Entrega del Laboratorio 4

- [ ] Métricas de validation y test (accuracy + macro-F1)
- [ ] Matriz de confusión
- [ ] 20 errores etiquetados con taxonomía + acción recomendada
- [ ] Checkpoint guardado y demo Gradio funcionando localmente
- [ ] `git commit -m "feat: complete finetuning lab"`

➡️ Continúa con el [**proyecto final**](../proyecto/README.md): baseline
TF-IDF, tu red propia y este Transformer, comparados bajo el mismo contrato.